# Chapitre 13 · Les données : construire le corpus francophone (solutions des exercices)

Ce notebook contient **uniquement les réponses aux quatre exercices** du
notebook du chapitre. Le code de la leçon, lui, vit dans le notebook du
chapitre et dans le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : le pacte
« IA débranchée » vaut aussi pour les corrigés.

## Mise en place (reprise de la leçon)

Le strict minimum repris de la leçon pour que les validations tournent :
hash, shingles, Jaccard, MinHash et statistiques de texte. Exécute cette
cellule d'abord.

In [ ]:
# Mise en place (reprise de la leçon) : le minimum pour exécuter les validations.
import re
import hashlib
from collections import Counter, defaultdict


def shingles(texte, n=3):
    """Ensemble des séquences de n mots consécutifs (fenêtre glissante)."""
    texte = texte.lower()
    texte = re.sub(r'[^0-9a-zàâäéèêëîïôöùûüç\s]', ' ', texte)
    mots = texte.split()
    ens = {' '.join(mots[i:i + n]) for i in range(len(mots) - n + 1)}
    return ens or {texte}


def jaccard(a, b):
    A, B = shingles(a), shingles(b)
    union = len(A | B)
    return len(A & B) / union if union else 0.0


def signature_minhash(sh, K=64):
    """K min-hashes : la signature du document. Forme : (K,)."""
    return [min(hash((s, k)) for s in sh) for k in range(K)]


MOTS_OUTILS = {
    'le', 'la', 'les', 'de', 'des', 'du', 'un', 'une', 'et', 'à', 'en',
    'que', 'qui', 'dans', 'pour', 'sur', 'au', 'aux', 'ce', 'il', 'elle',
    'se', 'ne', 'pas', 'son', 'sa', 'est',
}
PONCTUATION = set('.,;:!?-«»"\'()[]{}/\\|@#*+=%$')


def statistiques(texte):
    mots = texte.split()
    nb_mots = len(mots)
    ratio_ponct = sum(c in PONCTUATION for c in texte) / max(len(texte), 1)
    plus_frequent = Counter(m.lower() for m in mots).most_common(1)
    ratio_repetition = plus_frequent[0][1] / nb_mots if nb_mots else 1.0
    ratio_outils = sum(m.lower() in MOTS_OUTILS for m in mots) / nb_mots if nb_mots else 0.0
    return nb_mots, ratio_ponct, ratio_repetition, ratio_outils


print('mise en place OK')

### Exercice 1 · Déduplication exacte par hash — niveau ●

Refais le geste de la section 4 sans la regarder. Garde un document seulement
si son empreinte SHA256 n'a jamais été vue : ajoute alors le hash à `vus` et
le document à `gardes`. Rappel : un seul octet de différence change tout le
hash, la variante avec un « ! » doit donc survivre.

In [ ]:
def dedup_exacte(documents):
    vus, gardes = set(), []
    for doc in documents:
        h = hashlib.sha256(doc.encode('utf-8')).hexdigest()
        if h not in vus:        # jamais vu -> on garde
            vus.add(h)
            gardes.append(doc)
        # déjà vu -> doublon exact, on jette
    return gardes


miroirs = [
    'Maître Corbeau, sur un arbre perché, tenait en son bec un fromage.',
    'La cigale, ayant chanté tout l\'été, se trouva fort dépourvue.',
    'Maître Corbeau, sur un arbre perché, tenait en son bec un fromage.',
    'Maître Corbeau, sur un arbre perché, tenait en son bec un fromage !',
    'La cigale, ayant chanté tout l\'été, se trouva fort dépourvue.',
]
uniques = dedup_exacte(miroirs)
print(f'{len(miroirs)} -> {len(uniques)} documents')

In [ ]:
# Validation : déduplication exacte.
assert len(uniques) == 3, f'attendu 3 documents uniques, obtenu {len(uniques)}'
assert uniques[0].startswith('Maître Corbeau'), 'l\'ordre de première apparition doit être conservé'
assert uniques[2].endswith('!'), 'un octet de différence (le « ! ») doit suffire à garder le document'
print('Dédup exacte OK : 5 -> 3, les copies parfaites partent, la variante à un octet reste')

### Exercice 2 · La règle des mots-outils — niveau ●

Complète le filtre qualité de la section 6 : rejette un document, avec la
raison `'trop_peu_de_mots_outils'`, si son ratio de mots-outils est **sous**
le seuil `min_outils`. Un texte humain contient une proportion stable de
petits mots (« le », « de », « et »…) ; une liste de mots-clés n'en a
presque pas.

In [ ]:
def filtre_qualite(texte, min_mots=12, max_ponct=0.30,
                   max_repetition=0.20, min_outils=0.10):
    """Renvoie (gardé ?, liste des raisons de rejet)."""
    nb_mots, ratio_ponct, ratio_rep, ratio_outils = statistiques(texte)
    raisons = []
    if nb_mots < min_mots:
        raisons.append('trop_court')
    if ratio_ponct > max_ponct:
        raisons.append('trop_de_ponctuation')
    if ratio_rep > max_repetition:
        raisons.append('trop_repetitif')
    if ratio_outils < min_outils:
        raisons.append('trop_peu_de_mots_outils')
    return len(raisons) == 0, raisons


prose = 'La cigale, ayant chanté tout l\'été, se trouva fort dépourvue quand la bise fut venue.'
spam = 'achat vente promo solde discount deal offre bon plan code reduction cashback'
print('prose :', filtre_qualite(prose))
print('spam  :', filtre_qualite(spam))

In [ ]:
# Validation : règle des mots-outils.
ok_prose, raisons_prose = filtre_qualite(prose)
ok_spam, raisons_spam = filtre_qualite(spam)
assert ok_prose and raisons_prose == [], f'la prose devrait passer, raisons : {raisons_prose}'
assert not ok_spam, 'la liste de mots-clés devrait être rejetée'
assert raisons_spam == ['trop_peu_de_mots_outils'], f'raison attendue unique, obtenu : {raisons_spam}'
print('Filtre OK : la prose passe, la liste de mots-clés est rejetée pour', raisons_spam[0])

### Exercice 3 · La part effective du mélange — niveau ●●

Reprends le mélange de la section 8 : calcule la masse (documents × poids) de
chaque source, puis divise par la masse totale pour obtenir la part effective
de chacune. L'upsampling doit augmenter la part des fables au-delà de leur
volume brut.

In [ ]:
sources = {
    'fineweb_fr': {'documents': 8000, 'poids': 1.0},   # web, vu 1 fois
    'wikipedia_fr': {'documents': 1500, 'poids': 2.0}, # propre, vu 2 fois
    'fables': {'documents': 30, 'poids': 5.0},         # rare et soigné, vu 5 fois
}


def parts_du_melange(sources):
    """Part effective de chaque source = (documents x poids) / masse totale."""
    masse = {n: s['documents'] * s['poids'] for n, s in sources.items()}
    total = sum(masse.values())
    return {n: m / total for n, m in masse.items()}


parts = parts_du_melange(sources)
for nom, part in parts.items():
    print(f'{nom:14s} : {part * 100:5.1f}% du mélange')

In [ ]:
# Validation : mélange et upsampling.
part_fables = parts['fables']
part_fables_brut = 30 / (8000 + 1500 + 30)
assert abs(sum(parts.values()) - 1.0) < 1e-9, 'les parts doivent sommer à 1'
assert part_fables > part_fables_brut, 'l\'upsampling doit augmenter la part des fables'
assert abs(parts['wikipedia_fr'] - 3000 / 11150) < 1e-9, (
    f"part wikipedia attendue 26.9%, obtenu {parts['wikipedia_fr'] * 100:.1f}%")
print(f'Mélange OK : les fables passent de {part_fables_brut * 100:.2f}% (volume brut) '
      f'à {part_fables * 100:.2f}% (après upsampling ×5)')

### Exercice 4 · La vérification Jaccard du LSH — niveau ●●●

Le LSH de la section 5 ne fait que **proposer** des paires candidates.
Complète la vérification finale de `dedup_minhash` : pour chaque paire
candidate `(i, j)`, si la vraie Jaccard des deux documents dépasse `seuil`,
marque `j` comme doublon à retirer. LSH propose, Jaccard dispose.

In [ ]:
def dedup_minhash(documents, K=64, bandes=16, seuil=0.7):
    lignes = K // bandes
    sigs = [signature_minhash(shingles(d), K) for d in documents]

    seaux = defaultdict(list)
    for i, sig in enumerate(sigs):
        for b in range(bandes):
            bande = tuple(sig[b * lignes:(b + 1) * lignes])
            seaux[(b, hash(bande))].append(i)

    candidats = set()
    for ids in seaux.values():
        for a in range(len(ids)):
            for c in range(a + 1, len(ids)):
                candidats.add((ids[a], ids[c]))

    a_retirer = set()
    for i, j in candidats:
        if i in a_retirer or j in a_retirer:
            continue
        if jaccard(documents[i], documents[j]) > seuil:   # LSH propose, Jaccard dispose
            a_retirer.add(j)
    gardes = [d for k, d in enumerate(documents) if k not in a_retirer]
    return gardes, len(candidats)


originale = ('Rien ne sert de courir, il faut partir à point : le lièvre et la tortue '
             'en sont un témoignage que la fable raconte à tous les enfants.')
quasi = '[Accueil] ' + originale
autres = [
    'La raison du plus fort est toujours la meilleure, nous l\'allons montrer tout à l\'heure.',
    'Tout flatteur vit aux dépens de celui qui l\'écoute, cette leçon vaut bien un fromage sans doute.',
]
petits = [originale, quasi] + autres
gardes_lsh, n_cand = dedup_minhash(petits)
print(f'{len(petits)} -> {len(gardes_lsh)} documents ({n_cand} paire(s) candidate(s) testée(s))')

In [ ]:
# Validation : LSH + vérification Jaccard.
assert len(gardes_lsh) == 3, f'attendu 3 documents, obtenu {len(gardes_lsh)} : le quasi-doublon doit partir'
assert originale in gardes_lsh, 'l\'originale doit rester'
assert quasi not in gardes_lsh, 'la copie avec en-tête doit être retirée'
print('MinHash + LSH OK : le quasi-doublon est parti, les documents distincts restent')